# Iterative Research Agent — Search → Reflect → Search Loop

## Overview

This notebook builds a research agent that answers complex questions through an explicit multi-turn loop: plan, search, reflect on gaps, search again — repeating until confident. This is the foundational pattern behind the CVE scanner and earnings brief examples.

By making the loop explicit with a configurable max-iterations and confidence threshold, you can tune it for depth vs. cost.

### Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Interactive (Jupyter Notebook) |
| AgentCore components | AgentCore Gateway |
| Agentic framework | Strands Agents |
| LLM model | Anthropic Claude Sonnet 4 |
| Tutorial vertical | Cross-vertical |
| Example complexity | Advanced |
| SDK used | boto3, strands-agents |

### The Research Loop

```
Question
   │
   ▼
PLAN: Break into sub-questions
   │
   ▼
SEARCH: Execute highest-priority query ◀──────────────┐
   │                                                  │
   ▼                                                  │
REFLECT: What gaps remain?                            │
   ├── Gaps found → refine query, iterate (max N) ────┘
   └── Confident → synthesize final answer
```

### When Single-Shot Search Isn't Enough

Use this pattern when:
- The question requires comparing multiple sources
- Initial results reveal follow-up questions
- You need to reconcile conflicting information
- The topic has multiple dimensions that all matter

## Prerequisites

Complete `01-gateway-setup-and-raw-mcp` first and export the environment variables it prints.

### Required IAM permissions

~~~json
{
  "Effect": "Allow",
  "Action": "bedrock:InvokeModel",
  "Resource": "*"
}
~~~

## 1. Install Dependencies

In [ ]:
!pip install --upgrade -r requirements.txt --quiet

## 2. Configuration and Setup

In [ ]:
import os
import requests
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
GATEWAY_URL = os.environ["AGENTCORE_GATEWAY_URL"]
COGNITO_DOMAIN = os.environ["COGNITO_DOMAIN"]
CLIENT_ID = os.environ["COGNITO_CLIENT_ID"]
CLIENT_SECRET = os.environ["COGNITO_CLIENT_SECRET"]
SCOPE_STRING = os.environ["COGNITO_SCOPE"]
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"


def get_token():
    url = f"https://{COGNITO_DOMAIN}.auth.{REGION}.amazoncognito.com/oauth2/token"
    resp = requests.post(
        url,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={"grant_type": "client_credentials", "client_id": CLIENT_ID,
              "client_secret": CLIENT_SECRET, "scope": SCOPE_STRING}
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


def create_transport():
    return streamablehttp_client(GATEWAY_URL, headers={"Authorization": f"Bearer {get_token()}"})


# Tunable parameters
MAX_SEARCH_ITERATIONS = 4   # Maximum number of search rounds before forcing synthesis
CONFIDENCE_THRESHOLD = 0.8  # The agent uses this conceptually — expressed in the prompt

print(f"Max iterations  : {MAX_SEARCH_ITERATIONS}")
print(f"Configuration   : ✓")

## 3. Define the Iterative Research Agent

The system prompt explicitly encodes the plan → search → reflect → search loop with a stopping condition.

In [ ]:
ITERATIVE_RESEARCH_PROMPT = f"""You are a thorough research analyst with access to real-time web search.

RESEARCH LOOP (repeat up to {MAX_SEARCH_ITERATIONS} iterations):

STEP 1 — PLAN:
  Break the question into sub-questions. List them in priority order.
  Identify what you can answer confidently vs. what requires search.

STEP 2 — SEARCH:
  Execute the highest-priority unanswered sub-question as a web search.
  Keep queries under 200 characters.
  After each search, note: "Learned: <key finding>. Remaining gaps: <list>."

STEP 3 — REFLECT:
  Ask yourself: "Do I have enough to answer the original question confidently?"
  - If NO and iterations remain: formulate the next search based on gaps, go to STEP 2
  - If YES or iterations exhausted: go to STEP 4

STEP 4 — SYNTHESIZE:
  Write a comprehensive answer that:
  - Directly addresses the original question
  - Integrates findings from all search rounds
  - Cites URLs for all factual claims
  - Notes any remaining uncertainties

SHOW YOUR WORK: Make the plan, search queries, and reflections visible in your response.
This helps users understand the research process and verify the results.
"""

mcp_client = MCPClient(create_transport)
print("Iterative research agent ready ✓")

## 4. Research a Complex Question

This question requires multiple search angles — a single query can't answer it reliably.

In [ ]:
RESEARCH_QUESTION = """
What are the key differences between Claude Sonnet 4 and GPT-4.5 for enterprise code generation use cases,
and which one has better community adoption among developers as of 2026?
"""

print(f"Research question: {RESEARCH_QUESTION.strip()}")
print("-" * 60)
print("Running iterative research loop...")

with mcp_client:
    tools = mcp_client.list_tools_sync()
    model = BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.5, max_tokens=3000)
    agent = Agent(model=model, tools=tools, system_prompt=ITERATIVE_RESEARCH_PROMPT)
    response = agent(RESEARCH_QUESTION.strip())

print("\n" + "=" * 60)
print("RESEARCH REPORT")
print("=" * 60)
if hasattr(response, "message"):
    for block in response.message.get("content", []):
        if block.get("text"):
            print(block["text"])
else:
    print(str(response))

## 5. Try Questions That Require Multiple Search Angles

In [ ]:
multi_angle_questions = [
    "What regulatory changes in the EU in 2026 affect AI startup compliance, and what are the key deadlines?",
    "How has the adoption of Model Context Protocol changed agent development practices in 2025-2026?",
    "What are the current best practices for deploying LLM agents in production, based on recent industry reports?"
]

# Run one example (uncomment others as needed)
selected_question = multi_angle_questions[0]

print(f"Question: {selected_question}")
print("-" * 60)

with mcp_client:
    tools = mcp_client.list_tools_sync()
    model = BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.5, max_tokens=3000)
    agent = Agent(model=model, tools=tools, system_prompt=ITERATIVE_RESEARCH_PROMPT)
    response = agent(selected_question)

print("\n" + "=" * 60)
if hasattr(response, "message"):
    for block in response.message.get("content", []):
        if block.get("text"):
            print(block["text"])
else:
    print(str(response))

## 6. Tuning the Loop

Adjust `MAX_SEARCH_ITERATIONS` at the top of this notebook to control depth vs. latency/cost:

| Setting | Effect | Use when |
|:--------|:-------|:---------|
| 2 | Fast, shallow | Simple factual questions |
| 4 (default) | Balanced | Most research tasks |
| 6+ | Deep, thorough | Complex comparative analysis |

Each additional iteration costs one WebSearch call and one LLM call.

## Conclusion

In this notebook you built a research agent that:
- Plans its research before searching
- Reflects after each search to identify gaps
- Issues targeted follow-up queries until confident
- Synthesizes findings with cited sources

This is the foundational multi-turn pattern. Both the CVE scanner (search per item) and earnings brief (chained search) are specializations of this loop, tuned for their specific data structures.

### Summary: Three Multi-turn Patterns

| Pattern | Trigger | Example |
|:--------|:--------|:--------|
| Search per item | Fixed list of inputs | CVE scanner |
| Chained search | Each result shapes next query | Earnings brief |
| Iterative (this) | Reflect on gaps, search until confident | General deep research |